In [5]:
import pandas as pd
pd.set_option('display.max_columns', 500)

In [3]:
from Bio import SeqIO

def gbk_to_cds_fasta(gbk_file, output_fasta):
    with open(output_fasta, "w") as out_fasta:
        for record in SeqIO.parse(gbk_file, "genbank"):
            for feature in record.features:
                if feature.type == "CDS":
                    # Получаем locus_tag (основной ID)
                    locus_tag = feature.qualifiers.get("locus_tag", [None])[0]
                    if not locus_tag:
                        locus_tag = feature.qualifiers.get("protein_id", ["unknown_id"])[0]
                    
                    # Получаем gene (если есть)
                    gene = feature.qualifiers.get("gene", [""])[0]
                    
                    # Получаем product (описание)
                    product = feature.qualifiers.get("product", ["unknown product"])[0]
                    
                    # Получаем location (strand-aware)
                    start = feature.location.start + 1  # GBK использует 0-based, FASTA обычно 1-based
                    end = feature.location.end
                    strand = "+" if feature.location.strand >= 0 else "-"
                    location = f"{start}-{end}({strand})"
                    
                    # Получаем нуклеотидную последовательность CDS
                    cds_seq = feature.extract(record.seq)
                    
                    # Формируем FASTA-запись
                    header = f">{locus_tag}"
                    if gene:
                        header += f" gene:{gene}"
                    header += f" product:{product} location:{location}"
                    
                    out_fasta.write(f"{header}\n{cds_seq}\n")

# Пример использования:
gbk_to_cds_fasta("/mnt/SSD4TB/PROJECTS/kozyreva_works/110425_Vch_work/annotations/M1526_BC21_prokka/Vch_M1526_BC21.gbk", "/mnt/SSD4TB/PROJECTS/kozyreva_works/110425_Vch_work/annotations/M1526_BC21_prokka/Vch_M1526_BC21_cds_from_genomic.fna")

# Основной парсер

In [48]:
import pandas as pd

In [49]:
file_path = '/mnt/SSD4TB/PROJECTS/kozyreva_works/021125_SERRU_RNAseq_log/data/RISBM_SERRU_151.gff'

In [50]:
def parse_gff(file_path):
    data = []

    with open(file_path, 'r') as file:
        for line in file:
            if line.startswith('#'):
                continue

            columns = line.strip().split('\t')
            if len(columns) == 9:
                data.append(columns)
                
    columns = [
        'seqid', 'source', 'type', 'start', 'end', 
        'score', 'strand', 'phase', 'attributes'
    ]
    df = pd.DataFrame(data, columns=columns)
    
    df['start'] = df['start'].astype(int)
    df['end'] = df['end'].astype(int)
    df['score'] = pd.to_numeric(df['score'], errors='coerce')  # score может быть "."
    df['phase'] = pd.to_numeric(df['phase'], errors='coerce')  # phase может быть "."
    
    return df

In [51]:
gff_df = parse_gff(file_path)

In [52]:
gff_df

,seqid,source,type,start,end,score,strand,phase,attributes
0,C1_chromosome,Local,region,1,5076905,NaN,+,NaN,ID=C1_chromosome:1..5076905;Dbxref=taxon:61652...
1,C1_chromosome,.,gene,1,1392,NaN,+,NaN,ID=gene-pgaptmp_000001;Name=dnaA;gbkey=Gene;ge...
2,C1_chromosome,Protein Homology,CDS,1,1392,NaN,+,0.0,ID=cds-pgaptmp_000001;Parent=gene-pgaptmp_0000...
3,C1_chromosome,.,gene,1397,2497,NaN,+,NaN,ID=gene-pgaptmp_000002;Name=dnaN;gbkey=Gene;ge...
4,C1_chromosome,Protein Homology,CDS,1397,2497,NaN,+,0.0,ID=cds-pgaptmp_000002;Parent=gene-pgaptmp_0000...
...,...,...,...,...,...,...,...,...,...
9915,C2_plasmid1,Protein Homology,CDS,75670,77850,NaN,+,0.0,ID=cds-pgaptmp_004885;Parent=gene-pgaptmp_0048...
9916,C2_plasmid1,.,gene,77853,78506,NaN,+,NaN,ID=gene-pgaptmp_004886;Name=pgaptmp_004886;gbk...
9917,C2_plasmid1,Protein Homology,CDS,77853,78506,NaN,+,0.0,ID=cds-pgaptmp_004886;Parent=gene-pgaptmp_0048...
9918,C2_plasmid1,.,gene,78580,78810,NaN,+,NaN,ID=gene-pgaptmp_004887;Name=repC;gbkey=Gene;ge...


In [53]:
gff_df['type'].value_counts()

type
gene                4832
CDS                 4775
exon                 118
tRNA                  83
pseudogene            55
rRNA                  22
ncRNA                 10
riboswitch             9
sequence_feature       6
direct_repeat          5
region                 2
SRP_RNA                1
tmRNA                  1
RNase_P_RNA            1
Name: count, dtype: int64

## Работа с колонкой атрибутов

In [55]:
def extract_value(row, key):
    for cell in row:
        if pd.notna(cell) and cell.startswith(f"{key}="):
            return cell.split('=', 1)[1] 
    return None  

### Работа только с gene

In [56]:
gene_df = []

In [57]:
gene_df = gff_df[gff_df['type']=="gene"]

In [58]:
gene_attr = gene_df['attributes'].str.split(';', expand=True)

In [59]:
gene_main_attr = pd.DataFrame(columns=['ID', 'Name', 'gbkey', 'gene_biotype', 'locus_tag', 'old_locus_tag', 'partial'])

In [60]:
for idx, row in gene_attr.iterrows():
    id_value = extract_value(row, 'ID')
    name_value = extract_value(row, 'Name')
    gbkey_value = extract_value(row, 'gbkey')
    gene_biotype_value = extract_value(row, 'gene_biotype')
    locus_tag_value = extract_value(row, 'locus_tag')
    old_locus_tag_value = extract_value(row, 'old_locus_tag')
    partial_value = extract_value(row, 'partial')
    
    gene_main_attr.loc[idx] = [
        id_value, name_value, gbkey_value, gene_biotype_value,
        locus_tag_value, old_locus_tag_value, partial_value
    ]

In [61]:
gene_df = pd.concat([gene_df, gene_main_attr], axis=1)

In [62]:
#gene_df.sort_values(by='start', inplace=True)
gene_df.sort_values(by=['seqid', 'start'], inplace=True)

In [77]:
gene_df['gene_biotype'].unique()

array(['protein_coding', 'tRNA', 'ncRNA', 'rRNA', 'SRP_RNA', 'tmRNA',
       'RNase_P_RNA'], dtype=object)

In [63]:
gene_df

,seqid,source,type,start,end,score,strand,phase,attributes,ID,Name,gbkey,gene_biotype,locus_tag,old_locus_tag,partial
1,C1_chromosome,.,gene,1,1392,NaN,+,NaN,ID=gene-pgaptmp_000001;Name=dnaA;gbkey=Gene;ge...,gene-pgaptmp_000001,dnaA,Gene,protein_coding,pgaptmp_000001,None,None
3,C1_chromosome,.,gene,1397,2497,NaN,+,NaN,ID=gene-pgaptmp_000002;Name=dnaN;gbkey=Gene;ge...,gene-pgaptmp_000002,dnaN,Gene,protein_coding,pgaptmp_000002,None,None
5,C1_chromosome,.,gene,2659,3747,NaN,+,NaN,ID=gene-pgaptmp_000003;Name=recF;gbkey=Gene;ge...,gene-pgaptmp_000003,recF,Gene,protein_coding,pgaptmp_000003,None,None
7,C1_chromosome,.,gene,3766,6180,NaN,+,NaN,ID=gene-pgaptmp_000004;Name=gyrB;gbkey=Gene;ge...,gene-pgaptmp_000004,gyrB,Gene,protein_coding,pgaptmp_000004,None,None
9,C1_chromosome,.,gene,6304,7119,NaN,+,NaN,ID=gene-pgaptmp_000005;Name=yidA;gbkey=Gene;ge...,gene-pgaptmp_000005,yidA,Gene,protein_coding,pgaptmp_000005,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9910,C2_plasmid1,.,gene,73822,75030,NaN,+,NaN,ID=gene-pgaptmp_004883;Name=traW;gbkey=Gene;ge...,gene-pgaptmp_004883,traW,Gene,protein_coding,pgaptmp_004883,None,None
9912,C2_plasmid1,.,gene,75027,75677,NaN,+,NaN,ID=gene-pgaptmp_004884;Name=pgaptmp_004884;gbk...,gene-pgaptmp_004884,pgaptmp_004884,Gene,protein_coding,pgaptmp_004884,None,None
9914,C2_plasmid1,.,gene,75670,77850,NaN,+,NaN,ID=gene-pgaptmp_004885;Name=pgaptmp_004885;gbk...,gene-pgaptmp_004885,pgaptmp_004885,Gene,protein_coding,pgaptmp_004885,None,None
9916,C2_plasmid1,.,gene,77853,78506,NaN,+,NaN,ID=gene-pgaptmp_004886;Name=pgaptmp_004886;gbk...,gene-pgaptmp_004886,pgaptmp_004886,Gene,protein_coding,pgaptmp_004886,None,None


In [64]:
gene_df[gene_df['locus_tag']=='pgaptmp_004885']

,seqid,source,type,start,end,score,strand,phase,attributes,ID,Name,gbkey,gene_biotype,locus_tag,old_locus_tag,partial
9914,C2_plasmid1,.,gene,75670,77850,NaN,+,NaN,ID=gene-pgaptmp_004885;Name=pgaptmp_004885;gbk...,gene-pgaptmp_004885,pgaptmp_004885,Gene,protein_coding,pgaptmp_004885,None,None


In [94]:
gene_df[gene_df['seqid']=='iso2_chr1_manual_curated']['Name'].unique()

array(['dnaA', 'pgaptmp_000002', 'pgaptmp_000003', ..., 'pgaptmp_002217',
       'pgaptmp_002218', 'pgaptmp_002219'], dtype=object)

In [95]:
gene_df[(gene_df['seqid']=='iso2_chr1_manual_curated') & (gene_df['partial']=='true')]

,seqid,source,type,start,end,score,strand,phase,attributes,ID,Name,gbkey,gene_biotype,locus_tag,old_locus_tag,partial


In [65]:
gene_df.to_csv('/mnt/SSD4TB/PROJECTS/kozyreva_works/021125_SERRU_RNAseq_log/data/Serr_rubi_hybr_rotated_gene_gff_PARCED.csv', index=False)

### Работа только с CDS

In [66]:
cds_df = gff_df[gff_df['type']=="CDS"]

In [67]:
cds_attr = cds_df['attributes'].str.split(';', expand=True)

In [68]:
cds_main_attr = pd.DataFrame(columns=['ID', 'Name', 'gbkey', 'Parent', 'locus_tag', 'product', 'partial', 'pseudo', 'protein_id', 'go_process', 'go_function', 'Note'])

In [69]:
for idx, row in cds_attr.iterrows():
    id_value = extract_value(row, 'ID')
    name_value = extract_value(row, 'Name')
    gbkey_value = extract_value(row, 'gbkey')
    parent_value = extract_value(row, 'Parent')
    locus_tag_value = extract_value(row, 'locus_tag')
    product_value = extract_value(row, 'product')
    partial_value = extract_value(row, 'partial')
    pseudo_value = extract_value(row, 'pseudo')
    protein_id_value = extract_value(row, 'protein_id')
    go_process_value = extract_value(row, 'go_process')
    go_function_value = extract_value(row, 'go_function')
    note_value = extract_value(row, 'Note')
    
    cds_main_attr.loc[idx] = [
        id_value, name_value, gbkey_value, parent_value, locus_tag_value,
        product_value, partial_value, pseudo_value, protein_id_value,
        go_process_value, go_function_value, note_value
    ]

In [70]:
cds_df = pd.concat([cds_df, cds_main_attr], axis=1)

In [71]:
cds_df.sort_values(by=['seqid', 'start'], inplace=True)

In [72]:
cds_df

,seqid,source,type,start,end,score,strand,phase,attributes,ID,...,gbkey,Parent,locus_tag,product,partial,pseudo,protein_id,go_process,go_function,Note
2,C1_chromosome,Protein Homology,CDS,1,1392,NaN,+,0.0,ID=cds-pgaptmp_000001;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000001,...,CDS,gene-pgaptmp_000001,pgaptmp_000001,chromosomal replication initiator protein DnaA,None,None,extdb:pgaptmp_000001,"DNA replication initiation|0006270||IEA,regula...","DNA binding|0003677||IEA,DNA replication origi...",None
4,C1_chromosome,Protein Homology,CDS,1397,2497,NaN,+,0.0,ID=cds-pgaptmp_000002;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000002,...,CDS,gene-pgaptmp_000002,pgaptmp_000002,DNA polymerase III subunit beta,None,None,extdb:pgaptmp_000002,DNA replication|0006260||IEA,"DNA binding|0003677||IEA,DNA-directed DNA poly...",None
6,C1_chromosome,Protein Homology,CDS,2659,3747,NaN,+,0.0,ID=cds-pgaptmp_000003;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000003,...,CDS,gene-pgaptmp_000003,pgaptmp_000003,DNA replication/repair protein RecF,None,None,extdb:pgaptmp_000003,DNA repair|0006281||IEA,"single-stranded DNA binding|0003697||IEA,ATP b...",None
8,C1_chromosome,Protein Homology,CDS,3766,6180,NaN,+,0.0,ID=cds-pgaptmp_000004;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000004,...,CDS,gene-pgaptmp_000004,pgaptmp_000004,DNA topoisomerase (ATP-hydrolyzing) subunit B,None,None,extdb:pgaptmp_000004,DNA topological change|0006265||IEA,"DNA binding|0003677||IEA,DNA topoisomerase typ...",None
10,C1_chromosome,Protein Homology,CDS,6304,7119,NaN,+,0.0,ID=cds-pgaptmp_000005;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000005,...,CDS,gene-pgaptmp_000005,pgaptmp_000005,sugar-phosphatase,None,None,extdb:pgaptmp_000005,None,hydrolase activity|0016787||IEA,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9911,C2_plasmid1,Protein Homology,CDS,73822,75030,NaN,+,0.0,ID=cds-pgaptmp_004883;Parent=gene-pgaptmp_0048...,cds-pgaptmp_004883,...,CDS,gene-pgaptmp_004883,pgaptmp_004883,conjugal transfer protein TraW,None,None,extdb:pgaptmp_004883,None,None,None
9913,C2_plasmid1,Protein Homology,CDS,75027,75677,NaN,+,0.0,ID=cds-pgaptmp_004884;Parent=gene-pgaptmp_0048...,cds-pgaptmp_004884,...,CDS,gene-pgaptmp_004884,pgaptmp_004884,TraX-like protein,None,None,extdb:pgaptmp_004884,None,None,None
9915,C2_plasmid1,Protein Homology,CDS,75670,77850,NaN,+,0.0,ID=cds-pgaptmp_004885;Parent=gene-pgaptmp_0048...,cds-pgaptmp_004885,...,CDS,gene-pgaptmp_004885,pgaptmp_004885,DotA/TraY family protein,None,None,extdb:pgaptmp_004885,None,None,None
9917,C2_plasmid1,Protein Homology,CDS,77853,78506,NaN,+,0.0,ID=cds-pgaptmp_004886;Parent=gene-pgaptmp_0048...,cds-pgaptmp_004886,...,CDS,gene-pgaptmp_004886,pgaptmp_004886,ExcA,None,None,extdb:pgaptmp_004886,None,None,None


In [73]:
cds_df[cds_df['locus_tag']=='pgaptmp_004885']

,seqid,source,type,start,end,score,strand,phase,attributes,ID,...,gbkey,Parent,locus_tag,product,partial,pseudo,protein_id,go_process,go_function,Note
9915,C2_plasmid1,Protein Homology,CDS,75670,77850,NaN,+,0.0,ID=cds-pgaptmp_004885;Parent=gene-pgaptmp_0048...,cds-pgaptmp_004885,...,CDS,gene-pgaptmp_004885,pgaptmp_004885,DotA/TraY family protein,None,None,extdb:pgaptmp_004885,None,None,None


In [74]:
cds_df['product'].nunique()

3276

In [75]:
cds_df.to_csv('/mnt/SSD4TB/PROJECTS/kozyreva_works/021125_SERRU_RNAseq_log/data/Serr_rubi_hybr_rotated_genomic_cds_PARCED.csv', index=False)

In [59]:
fl_reg = cds_df[cds_df['product'].str.contains('transposase')]

In [34]:
fl_reg

,seqid,source,type,start,end,score,strand,phase,attributes,ID,Name,gbkey,Parent,locus_tag,product,partial,pseudo,protein_id,go_process,go_function,Note
22,BP_206,Protein Homology,CDS,12609,13559,NaN,+,0.0,ID=cds-pgaptmp_000011;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000011,extdb:pgaptmp_000011,CDS,gene-pgaptmp_000011,pgaptmp_000011,IS481-like element IS481 family transposase,None,None,extdb:pgaptmp_000011,DNA transposition|0006313||IEA,transposase activity|0004803||IEA,None
30,BP_206,Protein Homology,CDS,15645,16595,NaN,-,0.0,ID=cds-pgaptmp_000015;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000015,extdb:pgaptmp_000015,CDS,gene-pgaptmp_000015,pgaptmp_000015,IS481-like element IS481 family transposase,None,None,extdb:pgaptmp_000015,DNA transposition|0006313||IEA,transposase activity|0004803||IEA,None
89,BP_206,Protein Homology,CDS,52270,53220,NaN,+,0.0,ID=cds-pgaptmp_000044;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000044,extdb:pgaptmp_000044,CDS,gene-pgaptmp_000044,pgaptmp_000044,IS481-like element IS481 family transposase,None,None,extdb:pgaptmp_000044,DNA transposition|0006313||IEA,transposase activity|0004803||IEA,None
108,BP_206,Protein Homology,CDS,64726,65742,NaN,+,0.0,ID=cds-pgaptmp_000053;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000053,extdb:pgaptmp_000053,CDS,gene-pgaptmp_000053,pgaptmp_000053,IS110-like element IS1663 family transposase,None,None,extdb:pgaptmp_000053,DNA transposition|0006313||IEA,transposase activity|0004803||IEA,None
116,BP_206,Protein Homology,CDS,69197,70147,NaN,-,0.0,ID=cds-pgaptmp_000057;Parent=gene-pgaptmp_0000...,cds-pgaptmp_000057,extdb:pgaptmp_000057,CDS,gene-pgaptmp_000057,pgaptmp_000057,IS481-like element IS481 family transposase,None,None,extdb:pgaptmp_000057,DNA transposition|0006313||IEA,transposase activity|0004803||IEA,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7863,BP_206,Protein Homology,CDS,4061694,4062644,NaN,-,0.0,ID=cds-pgaptmp_003895;Parent=gene-pgaptmp_0038...,cds-pgaptmp_003895,extdb:pgaptmp_003895,CDS,gene-pgaptmp_003895,pgaptmp_003895,IS481-like element IS481 family transposase,None,None,extdb:pgaptmp_003895,DNA transposition|0006313||IEA,transposase activity|0004803||IEA,None
7869,BP_206,Protein Homology,CDS,4065107,4066057,NaN,+,0.0,ID=cds-pgaptmp_003898;Parent=gene-pgaptmp_0038...,cds-pgaptmp_003898,extdb:pgaptmp_003898,CDS,gene-pgaptmp_003898,pgaptmp_003898,IS481-like element IS481 family transposase,None,None,extdb:pgaptmp_003898,DNA transposition|0006313||IEA,transposase activity|0004803||IEA,None
7905,BP_206,Protein Homology,CDS,4086227,4087177,NaN,+,0.0,ID=cds-pgaptmp_003916;Parent=gene-pgaptmp_0039...,cds-pgaptmp_003916,extdb:pgaptmp_003916,CDS,gene-pgaptmp_003916,pgaptmp_003916,IS481-like element IS481 family transposase,None,None,extdb:pgaptmp_003916,DNA transposition|0006313||IEA,transposase activity|0004803||IEA,None
7911,BP_206,Protein Homology,CDS,4088337,4089287,NaN,+,0.0,ID=cds-pgaptmp_003919;Parent=gene-pgaptmp_0039...,cds-pgaptmp_003919,extdb:pgaptmp_003919,CDS,gene-pgaptmp_003919,pgaptmp_003919,IS481-like element IS481 family transposase,None,None,extdb:pgaptmp_003919,DNA transposition|0006313||IEA,transposase activity|0004803||IEA,None


# Считаем и выводим идентифицированные по аннотации гены

In [108]:
gene_df2 = gene_df
gene_df2['Name'] = gene_df['Name'].astype(str)
gene_df2[gene_df2['Name']=="None"].shape[0]

0

In [109]:
identified_genes = gene_df[~gene_df['Name'].str.contains('pgaptmp', na=False)][['Name','seqid','locus_tag']]
tmp_cds_df = cds_df[['locus_tag', 'product', 'partial','pseudo','protein_id','go_process','go_function','Note']] 						
tmp_identified = identified_genes.merge(tmp_cds_df, left_on='locus_tag',right_on='locus_tag', how='left')
identified_genes = tmp_identified[['Name','seqid','locus_tag','product','go_process','go_function','pseudo','partial','protein_id','Note']]
print (f'Всего записей по генам = {gene_df['Name'].shape[0]}')

print (f'Из них по идентифицированным генам = {identified_genes.shape[0]}')
print (f'Из записей по идентифицированным генам, по уникальным генам = {identified_genes['Name'].nunique()}')


#print (f'Из них по идентифицированным генам = {identified_genes.shape[0]-gene_df2[gene_df2['Name']=="None"].shape[0]}')
#print (f'Из записей по идентифицированным генам, по уникальным генам = {gene_df2['Name'][gene_df2['Name']!="None"].nunique()}')

Всего записей по генам = 2206
Из них по идентифицированным генам = 458
Из записей по идентифицированным генам, по уникальным генам = 442


In [110]:
identified_genes.to_csv('/mnt/SSD7TB/RUNS/kozyreva_ai/AKKM_ONT_MGI/annotation/PGAP_out/isolate2_mc_+100/isolate2_identified_genes_gff.csv', index=False)

In [111]:
identified_genes

,Name,seqid,locus_tag,product,go_process,go_function,pseudo,partial,protein_id,Note
0,dnaA,iso2_chr1_manual_curated,pgaptmp_000001,chromosomal replication initiator protein DnaA,"DNA replication initiation|0006270||IEA,regula...","DNA binding|0003677||IEA,DNA replication origi...",None,None,extdb:pgaptmp_000001,None
1,pheA,iso2_chr1_manual_curated,pgaptmp_000008,prephenate dehydratase,L-phenylalanine biosynthetic process|0009094||IEA,prephenate dehydratase activity|0004664||IEA,None,None,extdb:pgaptmp_000008,None
2,ligA,iso2_chr1_manual_curated,pgaptmp_000010,NAD-dependent DNA ligase LigA,"DNA replication|0006260||IEA,DNA repair|000628...","DNA binding|0003677||IEA,DNA ligase (NAD+) act...",None,None,extdb:pgaptmp_000010,None
3,pgsA,iso2_chr1_manual_curated,pgaptmp_000011,CDP-diacylglycerol--glycerol-3-phosphate 3-pho...,phospholipid biosynthetic process|0008654||IEA,CDP-diacylglycerol-glycerol-3-phosphate 3-phos...,None,None,extdb:pgaptmp_000011,None
4,eboE,iso2_chr1_manual_curated,pgaptmp_000016,metabolite traffic protein EboE,None,None,None,None,extdb:pgaptmp_000016,None
...,...,...,...,...,...,...,...,...,...,...
453,nadA,iso2_chr1_manual_curated,pgaptmp_002189,quinolinate synthase NadA,NAD biosynthetic process|0009435||IEA,quinolinate synthetase A activity|0008987||IEA...,None,None,extdb:pgaptmp_002189,None
454,nuoB,iso2_chr1_manual_curated,pgaptmp_002196,NADH-quinone oxidoreductase subunit NuoB,None,NADH dehydrogenase (ubiquinone) activity|00081...,None,None,extdb:pgaptmp_002196,None
455,argS,iso2_chr1_manual_curated,pgaptmp_002207,arginine--tRNA ligase,arginyl-tRNA aminoacylation|0006420||IEA,"arginine-tRNA ligase activity|0004814||IEA,ATP...",None,None,extdb:pgaptmp_002207,None
456,uvrA,iso2_chr1_manual_curated,pgaptmp_002215,excinuclease ABC subunit UvrA,nucleotide-excision repair|0006289||IEA,"DNA binding|0003677||IEA,ATP hydrolysis activi...",None,None,extdb:pgaptmp_002215,None


In [74]:
#iso1_id_genes = identified_genes['Name']

In [112]:
iso2_id_genes = identified_genes['Name']

In [113]:
iso2_id_genes.shape[0]

458

In [78]:
iso1_id_genes.shape[0]

453

In [72]:
name_counts = identified_genes['Name'].value_counts()
# Находим имена, которые встречаются больше одного раза - через будлевую маску true/false
non_unique_names = name_counts[name_counts > 1].index
non_unique_df = identified_genes[identified_genes['Name'].isin(non_unique_names)] # по идее это те гены, для которых есть две субъединицы или кофакторы

In [73]:
non_unique_df

,Name,seqid,locus_tag,product,go_process,go_function,pseudo,partial,protein_id,Note
22,ccsA,iso1_chr1_manual_corrected,pgaptmp_000098,cytochrome c biogenesis protein CcsA,cytochrome complex assembly|0017004||IEA,heme binding|0020037||IEA,None,None,extdb:pgaptmp_000098,None
40,def,iso1_chr1_manual_corrected,pgaptmp_000237,peptide deformylase,protein modification process|0036211||IEA,peptide deformylase activity|0042586||IEA,None,None,extdb:pgaptmp_000237,None
45,prfB,iso1_chr1_manual_corrected,pgaptmp_000248,peptide chain release factor 2,translational termination|0006415||IEA,translation release factor activity%2C codon s...,None,None,extdb:pgaptmp_000248,programmed frameshift
46,prfB,iso1_chr1_manual_corrected,pgaptmp_000248,peptide chain release factor 2,translational termination|0006415||IEA,translation release factor activity%2C codon s...,None,None,extdb:pgaptmp_000248,programmed frameshift
51,rrf,iso1_chr1_manual_corrected,pgaptmp_000264,NaN,NaN,NaN,NaN,NaN,NaN,NaN
86,thiE,iso1_chr1_manual_corrected,pgaptmp_000388,thiamine phosphate synthase,thiamine biosynthetic process|0009228||IEA,thiamine-phosphate diphosphorylase activity|00...,None,None,extdb:pgaptmp_000388,None
143,carB,iso1_chr1_manual_corrected,pgaptmp_000627,carbamoyl-phosphate synthase large subunit,metabolic process|0008152||IEA,"ATP binding|0005524||IEA,metal ion binding|004...",None,None,extdb:pgaptmp_000627,None
163,rho,iso1_chr1_manual_corrected,pgaptmp_000719,transcription termination factor Rho,DNA-templated transcription termination|000635...,"RNA binding|0003723||IEA,ATP binding|0005524||...",None,None,extdb:pgaptmp_000719,None
188,kdpA,iso1_chr1_manual_corrected,pgaptmp_000841,potassium-transporting ATPase subunit KdpA,potassium ion transport|0006813||IEA,P-type potassium transmembrane transporter act...,None,None,extdb:pgaptmp_000841,None
189,kdpB,iso1_chr1_manual_corrected,pgaptmp_000842,potassium-transporting ATPase subunit KdpB,potassium ion transport|0006813||IEA,"ATP binding|0005524||IEA,P-type potassium tran...",None,None,extdb:pgaptmp_000842,None


In [36]:
# non_unique_df.loc[non_unique_df['Name']=='dhaL'] # ладно, вариант ниже понятен, а это чё за хрень?

In [206]:
# non_unique_df.loc[non_unique_df['Name']=='mobA']

,Name,seqid,locus_tag,product,go_process,go_function,pseudo,partial,protein_id,Note
345,mobA,1,pgaptmp_000710,molybdenum cofactor guanylyltransferase MobA,Mo-molybdopterin cofactor biosynthetic process...,catalytic activity|0003824||IEA,None,None,extdb:pgaptmp_000710,None
1982,mobA,2,pgaptmp_004818,plasmid mobilization protein MobA,regulation of DNA-templated transcription|0006...,None,None,None,extdb:pgaptmp_004818,None


In [207]:
# non_unique_df.loc[non_unique_df['Name']=='alr'] # am i joke

,Name,seqid,locus_tag,product,go_process,go_function,pseudo,partial,protein_id,Note
45,alr,1,pgaptmp_000164,alanine racemase,"alanine metabolic process|0006522||IEA,peptido...",alanine racemase activity|0008784||IEA,None,None,extdb:pgaptmp_000164,None
120,alr,1,pgaptmp_000322,alanine racemase,"alanine metabolic process|0006522||IEA,peptido...",alanine racemase activity|0008784||IEA,None,None,extdb:pgaptmp_000322,None


In [208]:
non_unique_df.to_csv('/mnt/SSD4TB/PROJECTS/kozyreva_works/070225_Serratia_seq_data/twice_identified_genes_gff.csv', index=False)

# "__________________________________________________"

# Потоковый GFF парсер

In [ ]:
# Конфигурация
input_dir = "/mnt/SSD4TB/PROJECTS/kozyreva_works/110425_Vch_work/annotations/"  # Укажите вашу директорию
output_dir = "/mnt/SSD4TB/PROJECTS/kozyreva_works/110425_Vch_work/annotations_processed/"  # Директория для результатов

# Функции остаются теми же, что и в скрипте
def parse_gff(file_path):
    data = []
    with open(file_path, 'r') as file:
        for line in file:
            if line.startswith('#'):
                continue
            columns = line.strip().split('\t')
            if len(columns) == 9:
                data.append(columns)
                
    columns = ['seqid', 'source', 'type', 'start', 'end', 'score', 'strand', 'phase', 'attributes']
    df = pd.DataFrame(data, columns=columns)
    
    df['start'] = df['start'].astype(int)
    df['end'] = df['end'].astype(int)
    df['score'] = pd.to_numeric(df['score'], errors='coerce')
    df['phase'] = pd.to_numeric(df['phase'], errors='coerce')
    
    return df

def extract_value(row, key):
    for cell in row:
        if pd.notna(cell) and cell.startswith(f"{key}="):
            return cell.split('=', 1)[1] 
    return None

def process_gff_file(input_file, output_dir):
    sample_name = Path(input_file).stem
    sample_output_dir = os.path.join(output_dir, sample_name)
    os.makedirs(sample_output_dir, exist_ok=True)
    
    report_file = os.path.join(sample_output_dir, f"{sample_name}_gff_report.txt")
    
    # Красивый вывод в Jupyter
    display(Markdown(f"## Обработка файла: **{sample_name}**"))
    
    with open(report_file, 'w') as report:
        def log_print(*args, **kwargs):
            print(*args, **kwargs)
            print(*args, file=report, **kwargs)
        
        log_print(f"Processing file: {input_file}")
        
        # Парсинг и обработка GFF
        gff_df = parse_gff(input_file)
        
        # Гены
        gene_df = gff_df[gff_df['type']=="gene"]
        gene_attr = gene_df['attributes'].str.split(';', expand=True)
        gene_main_attr = pd.DataFrame(columns=['ID', 'Name', 'gbkey', 'gene_biotype', 'locus_tag', 'old_locus_tag', 'partial'])

        for idx, row in gene_attr.iterrows():
            gene_main_attr.loc[idx] = [
                extract_value(row, 'ID'),
                extract_value(row, 'Name'),
                extract_value(row, 'gbkey'),
                extract_value(row, 'gene_biotype'),
                extract_value(row, 'locus_tag'),
                extract_value(row, 'old_locus_tag'),
                extract_value(row, 'partial')
            ]

        gene_df = pd.concat([gene_df, gene_main_attr], axis=1)
        gene_df.sort_values(by=['seqid', 'start'], inplace=True)
        gene_output = os.path.join(sample_output_dir, f"{sample_name}_parced_gene_gff.csv")
        gene_df.to_csv(gene_output, index=False)
        
        # CDS
        cds_df = gff_df[gff_df['type']=="CDS"]
        cds_attr = cds_df['attributes'].str.split(';', expand=True)
        cds_main_attr = pd.DataFrame(columns=['ID', 'Name', 'gbkey', 'Parent', 'locus_tag', 'product', 'partial', 'pseudo', 'protein_id', 'go_process', 'go_function', 'Note'])

        for idx, row in cds_attr.iterrows():
            cds_main_attr.loc[idx] = [
                extract_value(row, 'ID'),
                extract_value(row, 'Name'),
                extract_value(row, 'gbkey'),
                extract_value(row, 'Parent'),
                extract_value(row, 'locus_tag'),
                extract_value(row, 'product'),
                extract_value(row, 'partial'),
                extract_value(row, 'pseudo'),
                extract_value(row, 'protein_id'),
                extract_value(row, 'go_process'),
                extract_value(row, 'go_function'),
                extract_value(row, 'Note')
            ]

        cds_df = pd.concat([cds_df, cds_main_attr], axis=1)
        cds_df.sort_values(by=['seqid', 'start'], inplace=True)
        cds_output = os.path.join(sample_output_dir, f"{sample_name}_parced_cds_gff.csv")
        cds_df.to_csv(cds_output, index=False)
        
        # Статистика
        display(Markdown("### Основная статистика:"))
        
        identified_genes = gene_df[~gene_df['Name'].str.contains('pgaptmp', na=False)][['Name','seqid','locus_tag']]
        tmp_cds_df = cds_df[['locus_tag', 'product', 'partial','pseudo','protein_id','go_process','go_function','Note']] 						
        tmp_identified = identified_genes.merge(tmp_cds_df, left_on='locus_tag',right_on='locus_tag', how='left')
        identified_genes = tmp_identified[['Name','seqid','locus_tag','product','go_process','go_function','pseudo','partial','protein_id','Note']]
        
        stats = {
            'Всего записей по генам': gene_df["Name"].shape[0],
            'Идентифицированные гены': identified_genes.shape[0],
            'Уникальные гены': identified_genes["Name"].nunique(),
            'Всего CDS': cds_df.shape[0],
            'Уникальные белковые продукты': cds_df["product"].nunique()
        }
        
        for k, v in stats.items():
            log_print(f"{k} = {v}")
            display(Markdown(f"- **{k}**: {v}"))
        
        identified_output = os.path.join(sample_output_dir, f"{sample_name}_identified_genes_gff.csv")
        identified_genes.to_csv(identified_output, index=False)
        
        # Неуникальные имена
        name_counts = identified_genes['Name'].value_counts()
        non_unique_names = name_counts[name_counts > 1].index
        non_unique_df = identified_genes[identified_genes['Name'].isin(non_unique_names)]
        non_unique_output = os.path.join(sample_output_dir, f"{sample_name}_twice_identified_genes_gff.csv")
        non_unique_df.to_csv(non_unique_output, index=False)
        
        display(f"\n Файл {sample_name} обработан. Результаты сохранены в: {sample_output_dir}")

# "_____________________ __Для CAI__ _____________________________"

# Работа по минимальному core-gene набору для CAI

In [103]:
core_genes_list = ['GroEL', 'DnaK', 'DnaJ', 'GrpE', 'DnaI', 'DnaB', 'DnaA', 'RecA', 'PriA', 'gyrA', 'gyrB', 
                   'topA', 'parC', 'parE', 'ung', 'nth', 'nusA', 'rpoA', 'rpoB', 'rpoC', 'rpoD', 'cca', 'iscS', 'obg',
                   'era','engA','smpB','rnc','pnp','GroES','SecA','SecY','SecE','aceE', 'aceF','aceF','lpdA','pdhA','pdhB']

In [117]:
result_dict = {}

In [118]:
counter=0
for index, row in identified_genes.iterrows():
    name_value = row['Name']
    if name_value in core_genes_list:
        if name_value not in result_dict:
            result_dict[name_value] = []
        result_dict[name_value].append(row['locus_tag'])
        counter=counter+1
        print(f"Значение '{name_value}' обнаружено в строке с locus_tag = {row['locus_tag']}")
print(f"Найдено '{counter}' core-генов")
print(f"________________________________________________")
for value in core_genes_list:
    if value not in result_dict: 
        print(f"Значение '{value}' не было найдено в датафрейме.")

Значение 'ung' обнаружено в строке с locus_tag = pgaptmp_000026
Значение 'gyrA' обнаружено в строке с locus_tag = pgaptmp_000780
Значение 'rpoA' обнаружено в строке с locus_tag = pgaptmp_001133
Значение 'nusA' обнаружено в строке с locus_tag = pgaptmp_001836
Значение 'pnp' обнаружено в строке с locus_tag = pgaptmp_001841
Значение 'smpB' обнаружено в строке с locus_tag = pgaptmp_001921
Значение 'rpoD' обнаружено в строке с locus_tag = pgaptmp_002421
Значение 'rnc' обнаружено в строке с locus_tag = pgaptmp_002485
Значение 'era' обнаружено в строке с locus_tag = pgaptmp_002486
Значение 'parE' обнаружено в строке с locus_tag = pgaptmp_002514
Значение 'parC' обнаружено в строке с locus_tag = pgaptmp_002515
Значение 'aceE' обнаружено в строке с locus_tag = pgaptmp_002531
Значение 'aceF' обнаружено в строке с locus_tag = pgaptmp_002532
Значение 'lpdA' обнаружено в строке с locus_tag = pgaptmp_002533
Значение 'gyrB' обнаружено в строке с locus_tag = pgaptmp_003025
Значение 'topA' обнаружено в 

In [137]:
print(f'Обнаруженные из заданного набора core-genes: \n {result_dict.items()}')

Обнаруженные из заданного набора core-genes: 
 dict_items([('ung', ['pgaptmp_000026']), ('gyrA', ['pgaptmp_000780']), ('rpoA', ['pgaptmp_001133']), ('nusA', ['pgaptmp_001836']), ('pnp', ['pgaptmp_001841']), ('smpB', ['pgaptmp_001921']), ('rpoD', ['pgaptmp_002421']), ('rnc', ['pgaptmp_002485']), ('era', ['pgaptmp_002486']), ('parE', ['pgaptmp_002514']), ('parC', ['pgaptmp_002515']), ('aceE', ['pgaptmp_002531']), ('aceF', ['pgaptmp_002532']), ('lpdA', ['pgaptmp_002533']), ('gyrB', ['pgaptmp_003025']), ('topA', ['pgaptmp_003095']), ('nth', ['pgaptmp_003306']), ('rpoC', ['pgaptmp_003383']), ('rpoB', ['pgaptmp_003384'])])


# Подготовка файла с эталонными белок-кодирующими генами для расчёта CAI

Оставляем только те из коровых генов, которые кодируют точно белки. Парсим полученную в ходе аннотации фасту с cds-nucl, выбираем из неё только белки, которые соответсвуют полученным коровым генам, их сохраняем в отдельную фасту. Эта фаста будет использоваться в качестве эталонного набора при расчёте CAI (codon adaptation index).  

In [126]:
from Bio import SeqIO

In [124]:
for key in list(result_dict.keys()):  # Используем list() для создания копии ключей
    locus_tags = result_dict[key]  # Список locus_tag для текущего ключа
    # Оставляем только те locus_tag, где есть хотя бы одна строка с gene_biotype == "protein_coding"
    filtered_locus_tags = [
        tag for tag in locus_tags
        if (gene_df[gene_df['locus_tag'] == tag]['gene_biotype'] == "protein_coding").any()
    ]
    result_dict[key] = filtered_locus_tags

In [ ]:
result_dict = {key: value for key, value in result_dict.items() if value}

In [125]:
result_dict

{'ung': ['pgaptmp_000026'],
 'gyrA': ['pgaptmp_000780'],
 'rpoA': ['pgaptmp_001133'],
 'nusA': ['pgaptmp_001836'],
 'pnp': ['pgaptmp_001841'],
 'smpB': ['pgaptmp_001921'],
 'rpoD': ['pgaptmp_002421'],
 'rnc': ['pgaptmp_002485'],
 'era': ['pgaptmp_002486'],
 'parE': ['pgaptmp_002514'],
 'parC': ['pgaptmp_002515'],
 'aceE': ['pgaptmp_002531'],
 'aceF': ['pgaptmp_002532'],
 'lpdA': ['pgaptmp_002533'],
 'gyrB': ['pgaptmp_003025'],
 'topA': ['pgaptmp_003095'],
 'nth': ['pgaptmp_003306'],
 'rpoC': ['pgaptmp_003383'],
 'rpoB': ['pgaptmp_003384']}

In [128]:
full_cds_fasta = "/mnt/SSD4TB/PROJECTS/kozyreva_works/040924_notebooks/CAI_codon_usage_test/VchM1504_Microbe/PGAP_out/VchM1504_Microbe_cds_from_genomic.fna"
core_cds_fasta = "/mnt/SSD4TB/PROJECTS/kozyreva_works/040924_notebooks/CAI_codon_usage_test/VchM1504_Microbe/VchM1504_Microbe_core_ref_cds.fna"

In [140]:
with open(core_cds_fasta, "w") as output_handle:
    for record in SeqIO.parse(full_cds_fasta, "fasta"):
       
        for key, values in result_dict.items():
            for value in values:
                if value in record.id: 
                    # Модифицируем заголовок, добавляя название ключа
                    record.id = f"{key}_{record.id}"
                    record.description = "" 
                    SeqIO.write(record, output_handle, "fasta")
                    break  # Прерываем внутренний цикл, чтобы избежать дублирования
    print (f'Эталонный набор белок-кодирующих core-генов для расчёта относительной адаптивности сохранен в {core_cds_fasta}')

Эталонный набор белок-кодирующих core-генов для расчёта относительной адаптивности сохранен в /mnt/SSD4TB/PROJECTS/kozyreva_works/040924_notebooks/CAI_codon_usage_test/VchM1504_Microbe/VchM1504_Microbe_core_ref_cds.fna


# gff_to_gtf

In [42]:
import pandas as pd
input_gff = "/mnt/SSD4TB/PROJECTS/kozyreva_works/021125_SERRU_RNAseq_log/data/RISBM_SERRU_151.gff"
output_gtf = "/mnt/SSD4TB/PROJECTS/kozyreva_works/021125_SERRU_RNAseq_log/data/tmp5_RISBM_SERRU_151.gtf"

In [43]:
# target_features = ['CDS', 'pseudogene', 'tRNA', 'rRNA', 'exon', 'ncRNA']
# target_features = ['CDS', 'tRNA', 'rRNA', 'ncRNA']
# target_features = ['gene', 'tRNA', 'rRNA', 'ncRNA']
target_features = ['gene']

In [44]:
def gff2gtf (target_features, input_gff, output_gtf):
    with open(input_gff, 'r') as infile, open(output_gtf, 'w') as outfile:
        for line in infile:
            if line.startswith('#'):
                continue
            columns = line.strip().split('\t')
            if len(columns) != 9:
                continue
            seqid, source, feature, start, end, score, strand, phase, attributes = columns
            if feature not in target_features:
                continue
            
            attr_dict = {}
            for attr in attributes.strip().split(';'):
                if '=' in attr:
                    key, value = attr.strip().split('=', 1)
                    attr_dict[key] = value
            gene_id = attr_dict.get('gene') or attr_dict.get('locus_tag') or attr_dict.get('ID') or "unknown"
            transcript_id = attr_dict.get('locus_tag')
            product_id = attr_dict.get('product') or "unknown"
            GO_terms = attr_dict.get('Ontology_term').strip().split(',') if attr_dict.get('Ontology_term') is not None else "unknown"

         #   print (gene_id, transcript_id, product_id, GO_terms, "\n")
            gtf_fields = [
                seqid,
                "RefSeq",
                "exon",
                start,
                end,
                score,
                strand,
                phase,
                f'gene_id "{gene_id}"; transcript_id "{transcript_id}"; product "{product_id}"; GO_terms "{GO_terms}" '
            ]
            outfile.write('\t'.join(gtf_fields) + '\n')
        print("File has converted successfully")     

In [45]:
gff2gtf(target_features, input_gff, output_gtf)

File has converted successfully
